[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Implement **multi-head attention with KV caching** for efficient autoregressive generation.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [8]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [9]:
import torch
import torch.nn as nn
import math

In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model 
        self.num_heads = num_heads
        self.d_k = d_model//num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        pass  # Initialize W_q, W_k, W_v, W_o

    def forward(self, x, cache=None):
        # 1. Project Q, K, V from x
        # 2. Reshape to multi-head: (B, num_heads, S, d_k)
        # 3. If cache exists, concat new K/V with cached K/V
        # 4. Compute attention (causal mask needed during prefill)
        # 5. Return (output, (K_all, V_all))
        
        B, S,_ = x.shape
        Q = self.W_q(x).reshape(B, S, self.num_heads, self.d_k).transpose(1, 2) # B, S, d_model
        K = self.W_k(x).reshape(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).reshape(B, S, self.num_heads, self.d_k).transpose(1, 2)
        if cache: # decode
            K_cache, V_cache = cache
            K_all = torch.cat([K_cache, K], dim = 2) # along seq len
            V_all = torch.cat([V_cache, V], dim = 2)

        else: # prefill
            K_all, V_all = K, V
            

        att = torch.matmul(Q, torch.transpose(K_all, -2,-1))/math.sqrt(self.d_k)
        if cache is None: # x contains entire prompt , apply mask and then get value vectors with this attn
            mask = torch.triu(torch.ones(S, S), diagonal=1)
            att.masked_fill_(mask.bool(), float('-inf'))


        attn = self.W_o(torch.matmul(torch.softmax(att, dim=-1), V_all).transpose(1 ,2).reshape(B, S, self.d_model))
        return attn, (K_all, V_all)
    #Prefill
# During prefill, the model sees a full sequence like ["The", "cat", "sat"] in one shot. It computes Q, K, and V for every position, and because the whole sequence is present, you need a causal mask so token 1 can’t look at token 2, token 2 can’t look at token 3, and so on.

# Decode
# During decode, you usually pass only the newest token, like "on", while reusing the cached K/V from the prompt and earlier generated tokens. Since the future tokens do not exist yet, the new token only attends to past tokens already in the cache, so the causality is enforced by the sequence order itself.

        pass

In [11]:
x[:, :4].shape

torch.Size([1, 4, 64])

In [12]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

Full output shape: torch.Size([1, 6, 64])
Cache K shape: torch.Size([1, 4, 4, 16])
Match: True


In [13]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')


🧪 Testing: KV Cache Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (no cache) (2.3ms)
  ✅ [2/5] Cache structure (1.5ms)
  ✅ [3/5] Decode step appends to cache (1.6ms)
  ✅ [4/5] Incremental decode matches full forward (2.1ms)
  ✅ [5/5] Gradient flow (18.1ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (25.7ms total)
  Progress saved. Run status() to see your dashboard.

